## Student Attendance Analytics with Pandas

In [ ]:
import pandas as pd
import numpy as np

#### Section 0: Data Loading

In [ ]:
df = pd.read_excel("attendance.xlsx")
df

#### Section 1: Data Inspection

In [ ]:
"""Q1. Display the first 10 rows, the shape of the DataFrame, and the data types of each column."""
print("First 10 rows:")
display(df.head(10))

print("\nDataFrame Shape:")
print(df.shape)

print("\nData Types:")
print(df.dtypes)

In [ ]:
"""Q2. Check for missing values in each column. Identify which columns have null values (especially Last Present Date with "Never") and report the count. """
# Q2: Check missing values
print("Missing values in each column:")
print(df.isnull().sum())

print("\nColumns with missing values:")
print(df.isnull().sum()[df.isnull().sum() > 0])

In [ ]:
"""Q3. Rename the columns to snake_case (e.g., Roll No → roll_no, Student Name → student_name, Attendance % → attendance_pct). Display the updated column list."""
# Q3: Rename columns to snake_case
df.rename(columns={
    'Roll No': 'roll_no',
    'Student Name': 'student_name',
    'Attendance %': 'attendance_pct'
}, inplace=True)

print("Updated column names:")
print(df.columns.tolist())

In [ ]:
"""Q4. Convert Attendance % from string format (e.g., "64%") to a numeric float column (0.64 or 64.0). Also convert Last Present Date to datetime, handling "Never" as NaT."""
# Q4: Convert Attendance % to numeric
df['attendance_pct'] = (
    df['attendance_pct']
    .astype(str)
    .str.replace('%', '', regex=False)
    .astype(float)
)

# Convert percentage values to decimal (e.g., 64% becomes 0.64)
df['attendance_pct'] = df['attendance_pct'] / 100

# Convert Last Present Date to datetime
df['last_present_date'] = pd.to_datetime(
    df['last_present_date'].replace('Never', pd.NaT),
    errors='coerce'
)

print(df[['attendance_pct', 'last_present_date']].dtypes)
print(df.head())

#### Section 2: Filtering & Selection

In [ ]:
"""Q5. Filter and display all students whose Status is "Critical". Sort them by Attendance % in ascending order."""
# Q5: Filter Critical students and sort by attendance
critical_students = df[df['Status'] == 'Critical'].sort_values(
    by='attendance_pct',
    ascending=True
)

print("Students with Critical Status:")
display(critical_students)

In [ ]:
"""Q6. Extract all students who have 0 present days (fully absent). List their roll numbers, names, and consecutive absences."""
fully_absent = df[df['Present Days'] == 0]

print("Fully Absent Students:")
display(fully_absent[['roll_no', 'student_name', 'Consecutive Absences']])

In [ ]:
"""Q7. Using .loc and .iloc, display:
(a) All "Safe" students using .loc
(b) The first 15 rows and columns 2-6 using .iloc """
# Q7(a): Display all Safe students using .loc
safe_students = df.loc[df['Status'] == 'Safe']

print("Safe Students:")
display(safe_students)

# Q7(b): Display the first 15 rows and columns 2-6 using .iloc
print("\nFirst 15 rows and columns 2-6:")
display(df.iloc[:15, 2:7])

In [ ]:
"""Q8. Find all students whose Consecutive Absences is greater than or equal to 5. Display their name, status, and last present date."""
# Q8: Students with at least 5 consecutive absences
students_absent = df[df['Consecutive Absences'] >= 5]

print("Students with 5 or More Consecutive Absences:")
display(students_absent[['student_name', 'Status', 'last_present_date']])

### Section 3: Grouping & Aggregation 

In [ ]:
"""Q9. Group the data by Status and compute: 
a) Count of students 
b) Average attendance %  
c) Maximum consecutive absences"""
# Q9: Grouping and aggregation
status_summary = df.groupby('Status').agg(
    student_count=('student_name', 'count'),
    average_attendance=('attendance_pct', 'mean'),
    max_consecutive_absences=('Consecutive Absences', 'max')
)

print("Status-wise Attendance Summary:")
display(status_summary)

In [ ]:
"""Q10. Create a new column Attendance Category:
≥ 90% → "Excellent"
75-89% → "Good"
60-74% → "Average"
< 60% → "Poor"
Then use groupby() to count students in each category."""
# Q10: Create Attendance Category

def categorize_attendance(attendance):
    if attendance >= 90:
        return "Excellent"
    elif attendance >= 75:
        return "Good"
    elif attendance >= 60:
        return "Average"
    else:
        return "Poor"

df['Attendance Category'] = df['attendance_pct'].apply(
    lambda x: categorize_attendance(x * 100)
)

print("Updated Student Data:")
display(df[['student_name', 'attendance_pct', 'Attendance Category']])

In [ ]:
"""Q11. Calculate the total number of classes conducted (sum of Total Classes) and the total present/absent counts across all students using .agg()."""
# Q11: Calculate totals using agg()

totals = df.agg({
    'Total Classes': 'sum',
    'Present Days': 'sum',
    'Absent': 'sum'
})

print("Total Classes, Present Days, and Absent:")
display(totals)

In [ ]:
"""Q12. Group by Status and use .agg() to find:
a)min, max, and mean of Present
b)sum of Absent"""
# Q12: Group by Status

status_stats = df.groupby('Status').agg({
    'Present Days': ['min', 'max', 'mean'],
    'Absent': 'sum'
})

print("Status-wise Statistics:")
display(status_stats)

### Section 4: Feature Engineering & Transformation

In [ ]:
"""Q13. Create a new column Risk Score using the formula:
Risk Score = (Absent * 0.4) + (Consecutive Absences * 0.6)
Display the top 10 students with the highest risk score."""
# Q13: Calculate Risk Score

df['Risk Score'] = (
    df['Absent'] * 0.4 +
    df['Consecutive Absences'] * 0.6
)

top_10_risk = df.sort_values(
    by='Risk Score',
    ascending=False
).head(10)

print("Top 10 Students with Highest Risk Score:")
display(top_10_risk)

In [ ]:
"""Q14. Add a column Classes Needed (Numeric) by extracting the numeric value from the Classes Needed (75%) column (e.g., "5 more" → 5, "✓ Met" → 0)."""
# Q14: Create Classes Needed (Numeric)

df['Classes Needed (Numeric)'] = (
    df['Classes Needed (75%)']
    .astype(str)
    .str.extract(r'(\d+)')[0]
    .fillna('0')
    .astype(int)
)

print("Updated Classes Needed:")
display(df[['student_name', 'Classes Needed (75%)',
            'Classes Needed (Numeric)']])

In [ ]:
"""Q15. Using apply() with a lambda function, create a column Attendance Flag that returns:
a)"Defaulter" if attendance < 75%
b)"Safe" otherwise """
# Q15: Create Attendance Flag

df['Attendance Flag'] = df['attendance_pct'].apply(
    lambda x: "Defaulter" if x * 100 < 75 else "Regular"
)

print("Attendance Flags:")
display(df[['student_name', 'attendance_pct', 'Attendance Flag']])

In [ ]:
"""Q16. Use pd.cut() to bin students into attendance ranges: [0-50, 50-75, 75-90, 90-100] and label them as ["Critical", "Warning", "Good", "Excellent"]."""
# Q16: Create Attendance Ranges

import pandas as pd

bins = [0, 50, 75, 90, 100]
labels = ["Critical", "Warning", "Good", "Excellent"]

df['Attendance Range'] = pd.cut(
    df['attendance_pct'] * 100,
    bins=bins,
    labels=labels,
    include_lowest=True
)

print("Attendance Ranges:")
display(df[['student_name', 'attendance_pct', 'Attendance Range']])

### Section 5: Advanced Analysis 

In [ ]:
"""Q17. Using pivot_table(), create a summary showing the count of students by Status and Attendance Category (created in Q10)."""
# Q17: Count students by Status and Attendance Category

pivot_summary = pd.pivot_table(
    df,
    index='Status',
    columns='Attendance Category',
    values='student_name',
    aggfunc='count',
    fill_value=0,
    observed=False
)

print("Student Count by Status and Attendance Category:")
display(pivot_summary)

In [ ]:
"""Q18. Identify duplicate student names (e.g., "ANANYA", "DISHA") and display all rows where the base name repeats. Use string manipulation to extract the base name."""
# Q18: Identify Duplicate Base Names

df['base_name'] = (
    df['student_name']
    .astype(str)
    .str.upper()
    .str.extract(r'^([A-Z]+)')[0]
)

duplicate_names = df[
    df['base_name'].duplicated(keep=False)
].sort_values('base_name')

print("Rows with Repeated Base Names:")
display(duplicate_names)

In [ ]:
""""Q19. Merge this dataset with a hypothetical Student_Details DataFrame containing Roll No, Branch, and Semester. Perform an inner join and display the merged result."""
# Q19: Create Student_Details DataFrame

Student_Details = pd.DataFrame({
    'Roll No': df['roll_no'],
    'Branch': ['CSE'] * len(df),
    'Semester': [3] * len(df)
})

# Merge using an inner join
merged_df = pd.merge(
    df,
    Student_Details,
    on='Roll No',
    how='inner'
)

print("Merged Student Details:")
display(merged_df)

In [ ]:
"""Q20. Export the following to separate CSV files:
(a) All "Critical" students
(b) Summary statistics grouped by Status
(c) Top 10 students by Risk Score"""
# Q20(a): Export Critical Students
critical_students = df[
    df['Status'].str.lower() == 'critical'
]
critical_students.to_csv('critical_students.csv', index=False)

# Q20(b): Export Summary Statistics Grouped by Status
status_summary = df.groupby('Status').agg(
    student_count=('student_name', 'count'),
    average_attendance=('attendance_pct', 'mean'),
    max_consecutive_absences=('Consecutive Absences', 'max')
)
status_summary.to_csv('status_summary.csv')

# Q20(c): Export Top 10 Students by Risk Score
top_10_students = df.sort_values(
    by='Risk Score',
    ascending=False
).head(10)

top_10_students.to_csv('top_10_risk_students.csv', index=False)

print("All three CSV files exported successfully!")